Example Notebook demonstrating scripted queries on the NRAO archive written by Mark Lacy and John Tobin.

In [1]:
import numpy as np
import pyvo as vo
import pandas as pd
#from astropy.coordinates import SkyCoord
import astropy.units as u
import urllib.request
import urllib.parse
from astroquery.simbad import Simbad
from astropy import units as u
from astropy.coordinates import SkyCoord


Lookup coordinates for a given target via Simbad or enter manually. If entering RA/Dec manually, a sexagesimal string should be used in a format that is compatible with the astropy SkyCoord library. A 'HH MM SS.SS' and 'DD MM SS.SS' format will work

In [2]:
result_table = Simbad.query_object("HOPS 203")
print(result_table)
ra=result_table['RA'][0]
dec=result_table['DEC'][0]
print(ra,dec)

          MAIN_ID                RA     ...     COO_BIBCODE     SCRIPT_NUMBER_ID
                              "h:m:s"   ...                                     
--------------------------- ----------- ... ------------------- ----------------
GBS-VLA J053622.86-064606.6 05 36 22.84 ... 2012AJ....144..192M                1
05 36 22.84 -06 46 06.2


Convert Sexagesimal string into a string holding the decimal coordinates.

In [3]:
c = SkyCoord(ra+' '+dec, unit=(u.hourangle, u.deg))
query_coords=c.to_string('decimal',precision=8)
ra_query=query_coords.split(' ')[0]
dec_query=query_coords.split(' ')[1]
print(ra_query,dec_query)

84.09516667 -6.76838889


First, run a query on the IRSA TAP service to make sure pyVO is behaving as expected 

In [4]:
service=vo.dal.TAPService("https://irsa.ipac.caltech.edu/TAP") 
search_radius=5.0 #in arcseconds
search_radius=search_radius/3600.0 # convert to degrees

In [5]:
result = service.search("""
           SELECT ra,dec,j_m,j_msigcom,h_m,h_msigcom,k_m,k_msigcom,ph_qual,cc_flg
           FROM fp_psc
           WHERE CONTAINS(POINT('ICRS',ra, dec), CIRCLE('ICRS',"""+ra_query+","+dec_query+","+str(search_radius)+"""))=1
    """)
tab = result.to_table()
print(tab)

    ra        dec      j_m   j_msigcom ...  k_m   k_msigcom ph_qual cc_flg
   deg        deg      mag      mag    ...  mag      mag                  
---------- ---------- ------ --------- ... ------ --------- ------- ------
 84.094957  -6.767589 16.212        -- ... 15.009     0.120     UCB    000


Now run the query on the NRAO TAP server

In [6]:
#access NRAO TAP server
service =vo.dal.TAPService("https://data-query.nrao.edu/tap") 

query="SELECT * FROM tap_schema.obscore WHERE CONTAINS(POINT('ICRS',s_ra,s_dec),CIRCLE('ICRS',"+ra_query+","+dec_query+","+str(search_radius)+"))=1" 
    
# Can also further select on other fields in the data, like instrument_name    
#query="SELECT * FROM tap_schema.obscore WHERE CONTAINS(POINT('ICRS',s_ra,s_dec),CIRCLE('ICRS',"+ra_query+","+dec_query+","+str(search_radius)+"))=1 and instrument_name='EVLA'" 

result = service.search(query)
output = result.to_table()

Print results

In [9]:
#display fields in result table
# print(result.fieldnames)

#print table
#use the .pprint(max_lines=-1) function to print the entire table regardless of lines/width
output.pprint(max_lines=-1,max_width=-1)
print(output)

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



Returned results show all the individual scans on the target from all executions. So, we need to determine the unique execution blocks and we can then calculate the time on source in each execution.

In [10]:
#print(output['obs_publisher_did','instrument_name','facility_name'])
#search returns all scans meeting query criteria 
#get unique EBs and their TOS
unique_EBs=np.unique(output['obs_publisher_did'])
print('Unique EBs: {}'.format(len(unique_EBs)))
for i in range(len(unique_EBs)):
    EB_index=np.where(output['obs_publisher_did'] == unique_EBs[i])
    total_tos_per_EB=np.sum(output['t_exptime'][EB_index[0]])
    freq_min,freq_max=output['freq_min'][EB_index[0][0]],output['freq_max'][EB_index[0][0]]
    instrument=output['instrument_name'][EB_index[0][0]]
    obs_id=output['obs_id'][EB_index[0][0]]

    print('Instrument: {}, EB: {}, TOS: {:0.1f}s, Nu_min - Nu_max: {:0.2f}-{:0.2f} GHz' \
         .format(instrument,unique_EBs[i],total_tos_per_EB,freq_min/1.0e9,freq_max/1.0e9))


Unique EBs: 69
Instrument: EVLA, EB: 16A-197.sb31830391.eb31893393.57446.070232847225, TOS: 1899.8s, Nu_min - Nu_max: 26.98-38.80 GHz
Instrument: EVLA, EB: 16A-197.sb32519316.eb32970950.57688.46910983796, TOS: 4487.8s, Nu_min - Nu_max: 27.10-38.80 GHz
Instrument: EVLA, EB: 16A-197.sb33086475.eb33205797.57758.13601657408, TOS: 4577.6s, Nu_min - Nu_max: 26.98-38.80 GHz
Instrument: EVLA, EB: 20B-122.sb38978088.eb39241748.59232.26183225695, TOS: 359.1s, Nu_min - Nu_max: 3.98-7.90 GHz
Instrument: EVLA, EB: 20B-122.sb39221023.eb39237373.59229.24359384259, TOS: 359.1s, Nu_min - Nu_max: 7.98-11.90 GHz
Instrument: VLA, EB: AB465_1_47128.22300_47128.46478.exp, TOS: 7200.0s, Nu_min - Nu_max: 1.42-1.42 GHz
Instrument: VLA, EB: AH218_1_46455.87300_46456.13897.exp, TOS: 1170.0s, Nu_min - Nu_max: 1.49-1.49 GHz
Instrument: VLA, EB: AH218_1_46842.00112_46842.22751.exp, TOS: 1140.0s, Nu_min - Nu_max: 1.67-1.67 GHz
Instrument: VLA, EB: AK373_1_49693.15395_49693.48427.exp, TOS: 6920.0s, Nu_min - Nu_max: 4

Now one can search those particular EB names for the data sets at https://data.nrao.edu in order to download the data products.